In [1]:
## importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

In [2]:
df = pd.read_excel('finalforms.xlsx')

In [3]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns',None)

In [4]:
print(df.shape)
# print(df.head())
df.columns[18]

(62, 104)


'Upon arrival, I prefer to interact with a human staff member rather than a digital system.'

In [5]:
# df

In [6]:
# Clean column names: strip spaces, replace line breaks, compress spaces
df.columns = (
    df.columns
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
    .str.replace(" +", " ", regex=True)
)

In [7]:
likert_cols = [
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system",
    "It feels easier to know what to do when a person guides me upon arrival",
    "Personal interaction with staff at arrival makes the dining experience feel more genuine",
    "It is easier to understand menu items when a person explains them",
    "I like when a staff member helps me explore the menu or suggests dishes",
    "I trust human staff more than a digital system to provide accurate menu information",
    "I enjoy hearing about dishes from a person rather than reading them on a screen",
    "When a digital system fails, having human staff as a fallback makes me feel supported",
    "I like having the option of human assistance while using a digital system",
    "It is easier to explain my dietary preferences or special needs to a person",
    "It is easier to change or adjust my order when interacting with human staff",
    "I feel reassured when a person confirms my customized order verbally",
    "It is easier to get assistance during the meal from human staff than from a digital system",
    "Human staff are better at judging the right time to bring the next course than digital systems",
    "I feel more comfortable communicating personal or dietary needs to a person than through a digital device",
    "Being checked on by human staff during the meal makes the experience feel more personal",
    "It is easier to ask follow-up questions about dishes when speaking to a person",
    "I trust human staff to handle payments more accurately than a digital system",
    "It is easier to clarify billing or payment questions with a person",
    "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system",
    "It is easier to understand the bill when a person explains what I am paying for",
    "Having a person handle payment makes me feel more confident and reassured about the transaction",
    "I feel more in control of my payment when interacting with a person rather than a digital system"
]


In [8]:
# Fix known naming mismatch FIRST
df = df.rename(columns={
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
})

# Then check if column is actually present in the dataframe
present = [c for c in likert_cols if c in df.columns]
missing = [c for c in likert_cols if c not in df.columns]

print("Missing:", missing)
print("Found:", len(present), "of", len(likert_cols))

Missing: []
Found: 23 of 23


In [9]:
# missing = [c for c in likert_cols if c not in df.columns]
# extra   = [c for c in df.columns if c in likert_cols]

# missing, len(missing)


In [10]:
# target = "Upon arrival, I prefer to interact with a human staff member rather than a digital system"

# [c for c in df.columns if "Upon arrival" in c]


In [11]:
# df = df.rename(columns={
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
# })


In [12]:
LIKERT_ORDER = {
    "strongly disagree": 1,
    "disagree": 2,
    "slightly disagree": 3,
    "slightly agree": 4,
    "agree": 5,
    "strongly agree": 6,
}


In [13]:
LIKERT_LABELS = {
    1: "strongly disagree",
    2: "disagree",
    3: "slightly disagree",
    4: "slightly agree",
    5: "agree",
    6: "strongly agree",
}


In [14]:
unique_responses = set()

for col in likert_cols:
    unique_responses.update(
        df[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
        .unique()
    )

sorted(unique_responses)




['agree',
 'disagree',
 'slightly agree',
 'slightly disagree',
 'strongly agree',
 'strongly disagree']

In [15]:
scale_keys = set(LIKERT_ORDER.keys())

unexpected = unique_responses - scale_keys
missing = scale_keys - unique_responses

unexpected, missing


(set(), set())

In [16]:
for col in likert_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )


In [17]:
for col in likert_cols:
    df[col] = df[col].map(LIKERT_ORDER)


In [18]:
#This computes how many missing values exist per Likert question column.

df[likert_cols].isna().sum()


Upon arrival, I prefer to interact with a human staff member rather than a digital system                    29
It feels easier to know what to do when a person guides me upon arrival                                      29
Personal interaction with staff at arrival makes the dining experience feel more genuine                     29
It is easier to understand menu items when a person explains them                                            29
I like when a staff member helps me explore the menu or suggests dishes                                      29
I trust human staff more than a digital system to provide accurate menu information                          30
I enjoy hearing about dishes from a person rather than reading them on a screen                              29
When a digital system fails, having human staff as a fallback makes me feel supported                        29
I like having the option of human assistance while using a digital system                               

In [19]:
# df = df.rename(columns={
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system.":
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system"
# })



p_vals = []

age_levels = df["Age band"].dropna().unique()

for q in likert_cols:
    groups = [
        df.loc[df["Age band"] == g, q].dropna()
        for g in age_levels
    ]
    # remove empty groups
    groups = [x for x in groups if len(x) > 0]

    if len(groups) >= 2:
        _, p = kruskal(*groups)
    else:
        p = np.nan

    p_vals.append(p)



In [20]:
# # groups

df["Age band"].unique()


p_vals

[0.9841868602151332,
 0.3672100543945057,
 0.44942925135423706,
 0.33474672508709263,
 0.5602583753826074,
 0.2728549461652404,
 0.9201442536349229,
 0.2214761522394015,
 0.7267154447969195,
 0.44729014473207596,
 0.30738158133793464,
 0.25424965203450745,
 0.5710202684116865,
 0.8200349854866675,
 0.9626835882599913,
 0.8715940421499073,
 0.735729792625291,
 0.06599720840899023,
 0.4837047807284802,
 0.6034295439017854,
 0.09958152579879757,
 0.05531405452019678,
 0.18328363999492445]

In [22]:
df_human_answered = df[df[likert_cols].notna().any(axis=1)]


In [23]:
# df_human_answered

In [24]:
age_counts = (
    df_human_answered["Age band"]
    .value_counts(dropna=False)
    .sort_index()
)

age_counts


Age band
18–24     1
25–34    14
35–44     7
45–54     6
55–64     3
65+       2
Name: count, dtype: int64

In [ ]:
# merging age bands

# Young:   18–34   → 1 + 14 = 15
# Middle:  35–54   → 7 + 6  = 13
# Older:   55+     → 3 + 2  = 5


In [25]:
def merge_age_band(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54"]:
        return "35–54"
    elif age in ["55–64", "65+"]:
        return "55+"
    else:
        return np.nan

df_human_answered = df_human_answered.copy()

df_human_answered["Age_3grp"] = df_human_answered["Age band"].apply(merge_age_band)





In [33]:
# df_human_answered["Age_3grp"]

In [ ]:
# df_human_answered

In [ ]:
# df_human_answered["Age_3grp"]

In [34]:
# # df_human_answered

# df_human_answered["Age_3grp"]

In [28]:


p_vals = []
H_vals = []

age_levels = df_human_answered["Age_3grp"].dropna().unique()

for q in likert_cols:
    groups = [
        df_human_answered.loc[
            df_human_answered["Age_3grp"] == g, q
        ].dropna()
        for g in age_levels
    ]
    
    # remove empty groups
    groups = [x for x in groups if len(x) > 0]

    if len(groups) >= 2:
        H, p = kruskal(*groups)
    else:
        H, p = np.nan, np.nan

    H_vals.append(H)
    p_vals.append(p)
    

# there is no sig diff withing people within age grpups who answered the quewstions

#



In [29]:

p_vals

[0.8636561885240785,
 0.09591470007962566,
 0.2066762780619738,
 0.8728431800260286,
 0.7122951368637528,
 0.2724364028680747,
 0.6492796100477998,
 0.1733598421444934,
 0.5400830766978082,
 0.20235750922938398,
 0.4632405646201818,
 0.126187768300768,
 0.4077028837199206,
 0.8651028558673521,
 0.6885060989125467,
 0.9848763742121578,
 0.4605886086870894,
 0.037922824241983726,
 0.1400719596151979,
 0.37927826611506105,
 0.11489300105902311,
 0.01767043449576508,
 0.07187865566496435]

In [35]:
# print(sorted(df["Age band"].dropna().unique()))
# print(sorted(df_human_answered["Age_3grp"].dropna().unique()))

In [36]:
results_df = pd.DataFrame({
    "question": likert_cols,
    "H_stat": H_vals,
    "p_value": p_vals
})

pd.set_option("display.max_colwidth", None)

results_df

results_df_sorted = results_df.sort_values("p_value", ascending=True)
results_df_sorted



,question,H_stat,p_value
21,Having a person handle payment makes me feel more confident and reassured about the transaction,8.071725,0.017670
17,I trust human staff to handle payments more accurately than a digital system,6.544404,0.037923
22,I feel more in control of my payment when interacting with a person rather than a digital system,5.265552,0.071879
1,It feels easier to know what to do when a person guides me upon arrival,4.688592,0.095915
20,It is easier to understand the bill when a person explains what I am paying for,4.327508,0.114893
11,I feel reassured when a person confirms my customized order verbally,4.139969,0.126188
18,It is easier to clarify billing or payment questions with a person,3.931198,0.140072
7,"When a digital system fails, having human staff as a fallback makes me feel supported",3.504772,0.173360
9,It is easier to explain my dietary preferences or special needs to a person,3.195439,0.202358
2,Personal interaction with staff at arrival makes the dining experience feel more genuine,3.153203,0.206676


In [ ]:
# Per-question age counts

# for q in likert_cols:
#     print("\n", q)
#     print(
#         df[df[q].notna()]["Age band"]
#         .value_counts()
#         .sort_index()
#     )


In [37]:
# for _, row in results_df_sorted.iterrows():
#     q = row["question"]
#     H = row["H_stat"]
#     p = row["p_value"]

#     print("\n" + "=" * 120)
#     print("QUESTION:")
#     print(q)
#     print("-" * 120)
#     print(f"H statistic: {H:.4f}")
#     print(f"p-value:     {p:.6f}")
#     print("-" * 120)

#     counts = (
#         df[q]
#         .value_counts(dropna=False)
#         .sort_index()
#         .reset_index()
#     )

#     counts.columns = ["response_code", "count"]
#     counts["response_text"] = counts["response_code"].map(LIKERT_LABELS)

#     # identify max count (ignore NaN response_code)
#     max_count = counts.loc[
#         counts["response_code"].notna(), "count"
#     ].max()

#     # mark the most frequent response(s)
#     counts["most_frequent"] = counts["count"].apply(
#         lambda x: "⭐ most frequent" if x == max_count else ""
#     )

#     print(counts)


In [ ]:
# # total count for human section only
# df_human_answered["Age band"].dropna().value_counts().sum()


In [38]:


# # Only correct non-missing p-values
# mask = results_df["p_value"].notna()

# # BH-adjusted p-values
# results_df.loc[mask, "p_adj_bh"] = multipletests(
#     results_df.loc[mask, "p_value"].values,
#     alpha=0.10,          # choose your FDR level here (e.g., 0.05 or 0.10)
#     method="fdr_bh"
# )[1]
# s
# # Flag significance at 10% FDR (change 0.10 to 0.05 if you want 5%)
# results_df["sig_fdr_10pct"] = results_df["p_adj_bh"] < 0.1

# # Sort by adjusted p-value (recommended for reporting)
# results_df_sorted = results_df.sort_values("p_adj_bh", ascending=True)

# results_df_sorted


In [ ]:
from scipy.stats import mannwhitneyu

age_col = "Age_3grp"

pairs = [
    ("18–34", "35–54"),
    ("18–34", "55+"),
    ("35–54", "55+")
]

for q in likert_cols:

    print(f"\n--- {q} ---")

    for g1, g2 in pairs:

        x = df_human_answered.loc[df_human_answered[age_col] == g1, q].dropna()
        y = df_human_answered.loc[df_human_answered[age_col] == g2, q].dropna()

        if len(x) == 0 or len(y) == 0:
            print(f"{g1} vs {g2}: NOT ENOUGH DATA")
            continue

        U, p = mannwhitneyu(x, y, alternative="two-sided")

        print(f"{g1} vs {g2} → p = {p:.4f}")


In [ ]:
from scipy.stats import mannwhitneyu

age_col = "Age_3grp"

pairs = [
    ("18–34", "35–54"),
    ("18–34", "55+"),
    ("35–54", "55+")
]

for q in likert_cols:

    print(f"\n--- {q} ---")

    for g1, g2 in pairs:

        x = df_human_answered.loc[df_human_answered[age_col] == g1, q].dropna()
        y = df_human_answered.loc[df_human_answered[age_col] == g2, q].dropna()

        if len(x) == 0 or len(y) == 0:
            print(f"{g1} vs {g2}: NOT ENOUGH DATA")
            continue

        U, p = mannwhitneyu(x, y, alternative="two-sided")

        print(f"{g1} vs {g2} → p = {p:.4f}")


In [ ]:
def print_crosstab_pretty(q, H, p, df_use):
    ct = pd.crosstab(df_use["Age_3grp"], df_use[q])
    ct = ct.reindex(columns=[1,2,3,4,5,6], fill_value=0).rename(columns=LIKERT_LABELS)

    headers = ["Age group"] + list(ct.columns)
    rows = [[idx] + [int(v) for v in ct.loc[idx].values] for idx in ct.index]

    # column widths
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]

    def fmt_row(r):
        return " | ".join(str(val).ljust(w) for val, w in zip(r, widths))

    print("\nQUESTION:")
    print(q)
    print(f"H = {H:.4f} | p = {p:.6f}\n")
    print(fmt_row(headers))
    print("-" * (sum(widths) + 3*(len(widths)-1)))
    for r in rows:
        print(fmt_row(r))

for _, row in results_df_sorted.iterrows():
    q = row["question"]
    print_crosstab_pretty(q, row["H_stat"], row["p_value"], df_human_answered)


## merging into two age groups

In [ ]:
# Younger: 18–34
# Older:   35+


In [ ]:
def merge_age_band_2(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    else:
        return "35+"

df_human_answered["Age_2grp"] = df_human_answered["Age band"].apply(merge_age_band_2)


In [ ]:
df_human_answered["Age_2grp"].value_counts()


In [ ]:
from scipy.stats import mannwhitneyu


mw_results = []

for q in likert_cols:
    g_young = df_human_answered.loc[
        df_human_answered["Age_2grp"] == "18–34", q
    ].dropna()

    g_old = df_human_answered.loc[
        df_human_answered["Age_2grp"] == "35+", q
    ].dropna()

    # Only run test if both groups have data
    if len(g_young) > 0 and len(g_old) > 0:
        U, p = mannwhitneyu(
            g_young,
            g_old,
            alternative="two-sided"
        )
    else:
        U, p = np.nan, np.nan

    mw_results.append({
        "question": q,
        "U_stat": U,
        "p_value": p,
        "n_18_34": len(g_young),
        "n_35_plus": len(g_old)
    })

mw_df = pd.DataFrame(mw_results)


In [ ]:
mw_df_sorted = mw_df.sort_values("p_value")
mw_df_sorted


In [ ]:
mw_df_sorted = mw_df.sort_values("p_value")
mw_df_sorted


In [ ]:
mask = mw_df["p_value"].notna()

mw_df.loc[mask, "p_adj_bh"] = multipletests(
    mw_df.loc[mask, "p_value"],
    method="fdr_bh"
)[1]

mw_df["sig_fdr_10pct"] = mw_df["p_adj_bh"] < 0.1


In [ ]:
mw_df.sort_values("p_adj_bh")


In [ ]:
mw_df["p_value"]


In [ ]:

def print_crosstab_pretty_mw(q, U, p, df_use, group_col="Age_2grp"):
    ct = pd.crosstab(df_use[group_col], df_use[q])
    ct = ct.reindex(columns=[1, 2, 3, 4, 5, 6], fill_value=0).rename(columns=LIKERT_LABELS)

    headers = ["Age group"] + list(ct.columns)
    rows = [[idx] + [int(v) for v in ct.loc[idx].values] for idx in ct.index]

    # column widths
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]

    def fmt_row(r):
        return " | ".join(str(val).ljust(w) for val, w in zip(r, widths))

    print("\nQUESTION:")
    print(q)
    print(f"U = {U:.4f} | p = {p:.6f}\n")
    print(fmt_row(headers))
    print("-" * (sum(widths) + 3 * (len(widths) - 1)))
    for r in rows:
        print(fmt_row(r))


In [ ]:
from scipy.stats import mannwhitneyu

mw_out = []

for q in likert_cols:
    g1 = df_human_answered.loc[df_human_answered["Age_2grp"] == "18–34", q].dropna()
    g2 = df_human_answered.loc[df_human_answered["Age_2grp"] == "35+", q].dropna()

    # only run if both groups have data
    if len(g1) == 0 or len(g2) == 0:
        continue

    U, p = mannwhitneyu(g1, g2, alternative="two-sided")
    mw_out.append((q, U, p))

# sort by p-value ascending
mw_out_sorted = sorted(mw_out, key=lambda x: x[2])

# print in sorted order
for q, U, p in mw_out_sorted:
    print_crosstab_pretty_mw(q, U, p, df_human_answered, group_col="Age_2grp")


In [ ]:

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

for q in likert_cols:
    df_human_answered[q] = pd.to_numeric(df_human_answered[q], errors="coerce")


age_groups = ["18–34", "35–54", "55+"]
age_pairs = list(combinations(age_groups, 2))

pairwise_human = []

for q in likert_cols:
    for g1, g2 in age_pairs:
        x = df_human_answered.loc[df_human_answered["Age_3grp"] == g1, q].dropna().to_numpy(dtype=float)
        y = df_human_answered.loc[df_human_answered["Age_3grp"] == g2, q].dropna().to_numpy(dtype=float)

        n1, n2 = len(x), len(y)

        if n1 < 3 or n2 < 3:
            U, p = np.nan, np.nan
        else:
            U, p = mannwhitneyu(x, y, alternative="two-sided")

        pairwise_human.append({
            "question": q,
            "group_1": g1,
            "group_2": g2,
            "n_1": n1,
            "n_2": n2,
            "U_stat": U,
            "p_value": p
        })

pairwise_human_df = pd.DataFrame(pairwise_human)
pairwise_human_df


In [ ]:
# print(df.columns.tolist())


In [ ]:
# #[col for col in likert_cols if col not in df.columns]

# df[likert_cols] = df[likert_cols].apply(lambda col: col.str.lower())


In [ ]:
# # 2. Make column names clean (optional but recommended)
# df.columns = df.columns.str.strip().str.rstrip(".")

# # 3. Check which names in likert_cols are incorrect
# not_in_df = [col for col in likert_cols if col not in df.columns]
# print("Columns not found:", not_in_df)

# # 4. Convert all Likert columns to lowercase text
# df[likert_cols] = df[likert_cols].apply(lambda col: col.str.lower())

# # 5. Map text → numeric values
# df[likert_cols] = df[likert_cols].replace(LIKERT_ORDER)


In [ ]:
# # Create a dictionary mapping long question text → short labels
# label_map = {
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system": "Arrival_Pref",
#     "It feels easier to know what to do when a person guides me upon arrival": "Guided_Ease",
#     "Personal interaction with staff at arrival makes the dining experience feel more genuine": "Arrival_Genuine",
#     "It is easier to understand menu items when a person explains them": "Menu_Understand",
#     "I like when a staff member helps me explore the menu or suggests dishes": "Menu_Help",
#     "I trust human staff more than a digital system to provide accurate menu information": "Menu_Trust",
#     "I enjoy hearing about dishes from a person rather than reading them on a screen": "Menu_Hear",
#     "When a digital system fails, having human staff as a fallback makes me feel supported": "Fallback_Support",
#     "I like having the option of human assistance while using a digital system": "Option_Help",
#     "It is easier to explain my dietary preferences or special needs to a person": "Dietary_Explain",
#     "It is easier to change or adjust my order when interacting with human staff": "Order_Adjust",
#     "I feel reassured when a person confirms my customized order verbally": "Order_Reassured",
#     "It is easier to get assistance during the meal from human staff than from a digital system": "Meal_Assist",
#     "Human staff are better at judging the right time to bring the next course than digital systems": "Timing_Judgement",
#     "I feel more comfortable communicating personal or dietary needs to a person than through a digital device": "Comfort_Dietary",
#     "Being checked on by human staff during the meal makes the experience feel more personal": "Checked_On",
#     "It is easier to ask follow-up questions about dishes when speaking to a person": "FollowUp_Questions",
#     "I trust human staff to handle payments more accurately than a digital system": "Pay_Trust",
#     "It is easier to clarify billing or payment questions with a person": "Pay_Clarify",
#     "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system": "Pay_Issue",
#     "It is easier to understand the bill when a person explains what I am paying for": "Bill_Understand",
#     "Having a person handle payment makes me feel more confident and reassured about the transaction": "Pay_Confident",
#     "I feel more in control of my payment when interacting with a person rather than a digital system": "Pay_Control"
# }

# corr_short = corr.rename(index=label_map, columns=label_map)
# corr_short


# plt.figure(figsize=(14, 12))

# plt.imshow(corr_short, cmap='coolwarm', vmin=-1, vmax=1)
# plt.colorbar(label="Spearman Correlation")

# # Set axis labels
# plt.xticks(np.arange(len(corr_short.columns)), corr_short.columns, rotation=90, fontsize=8)
# plt.yticks(np.arange(len(corr_short.index)), corr_short.index, fontsize=8)

# plt.title("Heatmap of Spearman Correlations (Short Labels)", fontsize=14)
# plt.tight_layout()
# plt.show()


In [ ]:
# INV_LIKERT_ORDER = {v: k for k, v in LIKERT_ORDER.items()}


# # AGREE = ["slightly agree", "agree", "strongly agree"]
# # DISAGREE = ["strongly disagree", "disagree", "slightly disagree"]


In [ ]:
# INV_LIKERT_ORDER

In [ ]:
# likert_cols1 = [
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system.",
#     "It feels easier to know what to do when a person guides me upon arrival",
#     "Personal interaction with staff at arrival makes the dining experience feel more genuine",
#     "It is easier to understand menu items when a person explains them",  
#     "I like when a staff member helps me explore the menu or suggests dishes",
#     "I trust human staff more than a digital system to provide accurate menu information",
#     "I enjoy hearing about dishes from a person rather than reading them on a screen",
#     "When a digital system fails, having human staff as a fallback makes me feel supported",
#     "I like having the option of human assistance while using a digital system",
#     "It is easier to explain my dietary preferences or special needs to a person",
#     "It is easier to change or adjust my order when interacting with human staff",
#     "I feel reassured when a person confirms my customized order verbally",  
#     "It is easier to get assistance during the meal from human staff than from a digital system",
#     "Human staff are better at judging the right time to bring the next course than digital systems",
#     "I feel more comfortable communicating personal or dietary needs to a person than through a digital device",
#     "Being checked on by human staff during the meal makes the experience feel more personal",
#     "It is easier to ask follow-up questions about dishes when speaking to a person",
#     "I trust human staff to handle payments more accurately than a digital system",
#     "It is easier to clarify billing or payment questions with a person",
#     "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system",
#     "It is easier to understand the bill when a person explains what I am paying for",
#     "Having a person handle payment makes me feel more confident and reassured about the transaction",
#     "I feel more in control of my payment when interacting with a person rather than a digital system"
# ]


In [ ]:
# p_vals = []

# for q in likert_cols1:
#     groups = [
#         df[df["Age_grouped"] == g][q]
#         for g in df["Age_grouped"].unique()
#     ]
#     _, p = kruskal(*groups)
#     p_vals.append(p)


In [ ]:
# df["Age_grouped"].value_counts()


In [ ]:
from scipy.stats import kruskal
import pandas as pd

results = []

for q in likert_cols1:
    # collect responses per age group
    groups = [
        df[df["Age_grouped"] == g][q]
        for g in df["Age_grouped"].unique()
    ]
    
    # Kruskal–Wallis test
    H, p = kruskal(*groups)
    
    results.append({
        "question": q,
        "H_statistic": H,
        "p_value": p
    })

results_df = pd.DataFrame(results)
results_df


In [ ]:
df[likert_cols1].dtypes


In [ ]:
for q in likert_cols1:
    df[q] = pd.to_numeric(df[q], errors="coerce")


In [ ]:
df

In [ ]:
for q in likert_cols1:
    df[q] = pd.to_numeric(df[q], errors="coerce")


In [ ]:
q = likert_cols1[0]

df.groupby("Age_grouped")[q].count()



In [ ]:
def recode_age_band(age):
    if age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54"]:
        return "35–54"
    elif age in ["55–64", "65+"]:
        return "55+"
    else:
        return pd.NA

df["Age_grouped"] = df["Age band"].apply(recode_age_band)


In [ ]:
# # AFTER you create df["Age2"] (18–34 vs 35+), run this HUMAN-only age test

# import numpy as np
# import pandas as pd
# from scipy.stats import mannwhitneyu
# from statsmodels.stats.multitest import multipletests

# # 1) Keep only rows with Age2
# human_df = df.dropna(subset=["Age2"]).copy()

# # 2) Ensure your HUMAN Likert columns are numeric
# # (If your df still has text, map -> numeric first, then force numeric)
# LIKERT_ORDER = {
#     "strongly disagree": 1,
#     "disagree": 2,
#     "slightly disagree": 3,
#     "slightly agree": 4,
#     "agree": 5,
#     "strongly agree": 6,
# }

# # If columns might still be strings, normalize + map
# human_df[likert_cols] = human_df[likert_cols].apply(lambda s: s.astype(str).str.strip().str.lower())
# human_df[likert_cols] = human_df[likert_cols].replace(LIKERT_ORDER)

# # Force numeric (fixes the isnan/type error)
# human_df[likert_cols] = human_df[likert_cols].apply(pd.to_numeric, errors="coerce")

# # 3) Mann–Whitney per question (because 2 age groups)
# pvals = []
# stats_u = []
# n_young = []
# n_old = []

# for q in likert_cols:
#     young = human_df.loc[human_df["Age2"] == "18–34", q].dropna()
#     old   = human_df.loc[human_df["Age2"] == "35+", q].dropna()

#     n_young.append(len(young))
#     n_old.append(len(old))

#     if len(young) >= 2 and len(old) >= 2:
#         u, p = mannwhitneyu(young, old, alternative="two-sided")
#     else:
#         u, p = np.nan, np.nan

#     stats_u.append(u)
#     pvals.append(p)

# # 4) FDR across all HUMAN items
# mask = ~pd.isna(pvals)
# rej = np.full(len(pvals), False, dtype=bool)
# p_fdr = np.full(len(pvals), np.nan, dtype=float)

# if mask.any():
#     rej_m, p_fdr_m, _, _ = multipletests(np.array(pvals)[mask], alpha=0.05, method="fdr_bh")
#     rej[mask] = rej_m
#     p_fdr[mask] = p_fdr_m

# # 5) Results table
# results = pd.DataFrame({
#     "Question": likert_cols,
#     "n_18_34": n_young,
#     "n_35plus": n_old,
#     "U": stats_u,
#     "p_raw": pvals,
#     "p_FDR": p_fdr,
#     "Significant_FDR": rej
# }).sort_values("p_FDR")

# results


In [ ]:
df["Age_grouped"].value_counts(dropna=False)


In [ ]:
df = df.dropna(subset=["Age_grouped"])


In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

pvals = []

for q in likert_cols:
    groups = [
        df.loc[df["Age_grouped"] == g, q].dropna()
        for g in ["18–34", "35–54", "55+"]
    ]
    
    groups = [g for g in groups if len(g) > 0]

    stat, p = kruskal(*groups)
    pvals.append(p)


In [ ]:
# from scipy.stats import kruskal
# from statsmodels.stats.multitest import multipletests
# import numpy as np
# import pandas as pd

# # 0) sanity: confirm groups exist
# print(df["Age_grouped"].value_counts(dropna=False))

# # 1) sanity: make sure likert columns are numeric
# df[likert_cols] = df[likert_cols].apply(pd.to_numeric, errors="coerce")

# pvals = []
# Hstats = []
# n_by_group = []   # store n per age group for each question

# age_levels = ["18–34", "35–54", "55+"]

# for q in likert_cols:
#     grp_series = []
#     ns = {}

#     for g in age_levels:
#         s = df.loc[df["Age_grouped"] == g, q].dropna()
#         ns[g] = len(s)
#         if len(s) > 0:
#             grp_series.append(s)

#     # 2) require at least 2 groups with data
#     if len(grp_series) < 2:
#         Hstats.append(np.nan)
#         pvals.append(np.nan)
#         n_by_group.append(ns)
#         continue

#     # 3) optional: warn if any group is very small
#     # (doesn't stop the test, just flags it)
#     if min(ns.values()) < 3:
#         # you can print a warning or keep it quiet
#         # print(f"Warning: small n for '{q}': {ns}")
#         pass

#     H, p = kruskal(*grp_series)
#     Hstats.append(H)
#     pvals.append(p)
#     n_by_group.append(ns)

# # FDR (ignore NaNs)
# mask = ~pd.isna(pvals)
# rej = np.array([False]*len(pvals), dtype=bool)
# pvals_fdr = np.array([np.nan]*len(pvals), dtype=float)

# rej_sub, p_fdr_sub, _, _ = multipletests(np.array(pvals)[mask], alpha=0.05, method="fdr_bh")
# rej[mask] = rej_sub
# pvals_fdr[mask] = p_fdr_sub

# results = pd.DataFrame({
#     "Question": likert_cols,
#     "H": Hstats,
#     "p_raw": pvals,
#     "p_FDR": pvals_fdr,
#     "Significant_FDR": rej,
#     "n_18_34": [d["18–34"] for d in n_by_group],
#     "n_35_54": [d["35–54"] for d in n_by_group],
#     "n_55plus": [d["55+"] for d in n_by_group],
# }).sort_values("p_raw")

# results


In [ ]:
pvals

In [ ]:
rej, pvals_fdr, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")


In [ ]:
pd.set_option("display.max_colwidth", 200)   # or None for unlimited
pd.set_option("display.width", 200)


In [ ]:
results = pd.DataFrame({
    "Question": likert_cols,
    "p_raw": pvals,
    "p_FDR": pvals_fdr,
    "Significant": rej
})

results.sort_values("p_FDR")


In [ ]:
human_mask = df[likert_cols].notna().any(axis=1)
df_human = df.loc[human_mask]

df_human["Age_grouped"].value_counts(dropna=False)

In [ ]:
import pingouin as pg

# No shared concpetual similarity
#f α ≥  0.9225255607267622 → good construct

# keep question 1 seperate

# remaining 4 keep seperate


orange_cols = [
    # "I feel reassured when a person confirms my customized order verbally",
    
    "I trust human staff to handle payments more accurately than a digital system",
    "It is easier to understand the bill when a person explains what I am paying for",
    "Having a person handle payment makes me feel more confident and reassured about the transaction",
    "I feel more in control of my payment when interacting with a person rather than a digital system",    
]

# OPTIONAL: check that all names actually exist in df
missing = [c for c in orange_cols if c not in df.columns]
print("Missing:", missing)  # should be [] before proceeding

# Now compute alpha (assuming these columns are already numeric 1–6)
alpha, ci = pg.cronbach_alpha(data=df[orange_cols])
print("Orange Alpha:", alpha)
print("95% CI:", ci)


#Reassurance & Human Accuracy Trust
#"I feel reassured when a person confirms my customized order verbally"

#Menu Guidance & Staff Assistance
#"I trust human staff to handle payments more accurately than a digital system",
#"It is easier to understand the bill when a person explains what I am paying for",
#Having a person handle payment makes me feel more confident and reassured about the transaction",
#"I feel more in control of my payment when interacting with a person rather than a digital system"


In [ ]:
counts = (
    df
    .groupby("Age_grouped")[likert_cols1]
    .count()
)

counts


In [ ]:
# from here on the functionalized code runs

from scipy.stats import kruskal

In [ ]:
import pandas as pd

def age_group_counts_for_items(
    df,
    likert_cols1,
    age_col="Age band"
):
    """
    Recode age bands into grouped age categories
    and count non-NA responses per age group
    for a set of Likert items.
    """

    # --- Step 1: recode age ---
    def recode_age(age):
        if age in ["18–24", "25–34"]:
            return "18–34"
        elif age in ["35–44", "45–54"]:
            return "35–54"
        elif age in ["55–64", "65+"]:
            return "55+"
        else:
            return pd.NA

    df = df.copy()
    df["Age_grouped"] = df[age_col].apply(recode_age)

    # --- Step 2: count valid responses ---
    counts = (
        df
        .groupby("Age_grouped")[likert_cols1]
        .count()
    )

    return counts


In [ ]:
counts = age_group_counts_for_items(df, likert_cols1)
counts


In [ ]:
import pandas as pd
from scipy.stats import kruskal


pd.set_option("display.max_colwidth", None)


def run_kruskal_all_items_no_min_n(
    df,
    likert_cols,
    age_col="Age band"
):
    """
    Runs Kruskal–Wallis for EVERY item across 3 age groups (18–34, 35–54, 55+),
    without enforcing any minimum n.

    If any group is empty (n=0), Kruskal cannot run -> returns NaN for H and p_value,
    but DOES NOT skip the item (it still appears in the results).
    """

    # --- recode age into 3 buckets ---
    def recode_age(age):
        if age in ["18–24", "25–34"]:
            return "18–34"
        elif age in ["35–44", "45–54"]:
            return "35–54"
        elif age in ["55–64", "65+"]:
            return "55+"
        else:
            return pd.NA

    df2 = df.copy()
    df2["Age_grouped"] = df2[age_col].apply(recode_age)

    results = []

    for item in likert_cols:
        # Keep only rows where age + item exist
        sub = df2.dropna(subset=["Age_grouped", item])[["Age_grouped", item]].copy()

        # Convert Likert values to numeric safely
        sub[item] = pd.to_numeric(sub[item], errors="coerce")
        sub = sub.dropna(subset=[item])

        # Split by group
        g18_34 = sub.loc[sub["Age_grouped"] == "18–34", item]
        g35_54 = sub.loc[sub["Age_grouped"] == "35–54", item]
        g55p   = sub.loc[sub["Age_grouped"] == "55+", item]

        n18, n35, n55 = len(g18_34), len(g35_54), len(g55p)

        # If any group is empty, Kruskal can't run
        if n18 == 0 or n35 == 0 or n55 == 0:
            results.append({
                "item": item,
                "test": "Kruskal–Wallis",
                "H": float("nan"),
                "p_value": float("nan"),
                "n_18_34": n18,
                "n_35_54": n35,
                "n_55p": n55,
                "note": "Cannot run: at least one group has n=0"
            })
            continue

        # Run Kruskal–Wallis even if n is small
        H, p = kruskal(g18_34, g35_54, g55p)

        results.append({
            "item": item,
            "test": "Kruskal–Wallis",
            "H": float(H),
            "p_value": float(p),
            "n_18_34": n18,
            "n_35_54": n35,
            "n_55p": n55,
            "note": ""
        })

    return pd.DataFrame(results).sort_values("p_value", na_position="last").reset_index(drop=True)


In [ ]:
results = run_kruskal_all_items_no_min_n(df, likert_cols1, age_col="Age band")
results


In [ ]:
import numpy as np

def add_fdr_bh(results_df, p_col="p_value", alpha=0.05):
    df = results_df.copy()

    # Extract p-values
    pvals = df[p_col].values
    mask = ~np.isnan(pvals)

    pvals_nonan = pvals[mask]
    m = len(pvals_nonan)

    # Rank p-values
    order = np.argsort(pvals_nonan)
    ranked_pvals = pvals_nonan[order]

    # Benjamini–Hochberg formula
    qvals = ranked_pvals * m / (np.arange(1, m + 1))

    # Enforce monotonicity
    qvals = np.minimum.accumulate(qvals[::-1])[::-1]
    qvals = np.clip(qvals, 0, 1)

    # Put back into original order
    qvals_full = np.full_like(pvals, np.nan, dtype=float)
    qvals_full[mask] = qvals[np.argsort(order)]

    df["q_value_BH"] = qvals_full
    df["sig_FDR_0.05"] = df["q_value_BH"] < alpha

    return df


In [ ]:
results_fdr = add_fdr_bh(results)
results_fdr


In [ ]:
# does age make a difference in how people respond to these human interaction questions
# For each question, you look at:

# 🔹 p-value

# p < 0.05 → evidence that age groups differ for this question

# p ≥ 0.05 → no convincing evidence of age differences


#Even if one question has p < 0.05:

# That could be luck

# Especially when you test many questions

# That’s why you then did FDR.

## Question to answer from kruskal wallis test

## Do multiple groups come from the same distribution, or not?

## Null hypothesis: Nothing systematic is going on meaning all athe age groups have same response distribution for all the questions in the human modality section

## Alternate hypothesis: all athe age groups have some significant difference in their response distribution for all the questions in the human modality section


## Statistical test level hypothesis
## They are defined by Kruskal–Wallis
## They exist every time you run the test
## They apply per question

# H0 = The distributions of Likert responses are identical across age groups.
# H1 = At least one age group differs from the others

In [ ]:
## An exploratory research question framed as a hypothesis for structure.

In [ ]:
#🧠 Analogy

#Statistical H0/H1 = the engine of the test

#Exploratory hypothesis = the destination of the study

In [ ]:
col = "I like when a staff member helps me explore the menu or suggests dishes"

s = df[col]

print("dtype:", s.dtype)
print("Total rows:", len(s))
print("NaN count:", s.isna().sum())
print("Blank strings:", (s.astype(str).str.strip() == "").sum())
print("Top raw values:")
print(s.astype(str).str.strip().value_counts(dropna=False).head(10))


In [ ]:
df[likert_cols1] 

In [ ]:
# for col in likert_cols1:
#     df[col] = (
#         df[col]
#         .astype(str)
#         .str.strip()
#         .str.lower()
#         .map(INV_LIKERT_ORDER)
#     )


# df.columns = df.columns.str.strip().str.rstrip(".")



In [ ]:
df

In [ ]:
LIKERT_MAP_NUM_TO_STR = {
    1: "strongly disagree",
    2: "disagree",
    3: "slightly disagree",
    4: "slightly agree",
    5: "agree",
    6: "strongly agree",
}


In [ ]:
for col in likert_cols1:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")   # ensure numeric
        .map(LIKERT_MAP_NUM_TO_STR)               # convert to text
    )


In [ ]:
df["I trust human staff to handle payments more accurately than a digital system"].value_counts(dropna=False)


In [ ]:
# print(df[likert_cols1].head())
# print(df[likert_cols1].dtypes)
# # 

In [ ]:
# print(df["I trust human staff to handle payments more accurately than a digital system"].value_counts(dropna=False))


In [ ]:
# df

In [ ]:
# df['Gender'].value_counts()

In [ ]:
# df['Age band'].value_counts()

In [ ]:

age = "Age band"
modality = "When dining at a restaurant, what is your preferred way to place an order?"


In [ ]:
AGREE = ["slightly agree", "agree", "strongly agree"]
DISAGREE = ["strongly disagree", "disagree", "slightly disagree"]

def compute_overall_sentiment(df, likert_col):
    # drop NA
    series = df[likert_col].dropna()

    total = series.shape[0]
    agree = series.isin(AGREE).sum()
    disagree = series.isin(DISAGREE).sum()

    agree_pct = agree / total * 100
    disagree_pct = disagree / total * 100

    return agree, disagree, total, agree_pct, disagree_pct


In [ ]:
for q in arrival_cols:
    agree, disagree, total, agree_pct, disagree_pct = compute_overall_sentiment(df, q)
    print(q)
    print(f"Responses: {total}")
    print(f"Agreement: {agree_pct:.1f}%  |  Disagreement: {disagree_pct:.1f}%\n")


In [ ]:
LIKERT_ORDER_PLOT = [
    "strongly disagree",
    "disagree",
    "slightly disagree",
    "slightly agree",
    "agree",
    "strongly agree",
]

AGREE = ["slightly agree", "agree", "strongly agree"]
DISAGREE = ["strongly disagree", "disagree", "slightly disagree"]

df = df.apply(lambda col: col.str.lower() if col.dtype == "object" else col)


def plot_checkbox_vs_likert(df, checkbox_col, likert_col,
                            title_prefix="", figsize=(10,4)):

    # --- STEP 1: explode checkbox column to long format ---
    cb = df[checkbox_col].fillna("").str.split(";")
    cb_exploded = cb.explode().str.strip()
    cb_exploded = cb_exploded[cb_exploded != ""]
    df_long = df.loc[cb_exploded.index].copy()
    df_long["checkbox_cat"] = cb_exploded

    # --- STEP 2: crosstab (COUNTS) ---
    ct = pd.crosstab(
        df_long["checkbox_cat"],
        df_long[likert_col]
    )

    # ensure all Likert categories exist & in right order
    for resp in LIKERT_ORDER_PLOT:
        if resp not in ct.columns:
            ct[resp] = 0
    ct = ct[LIKERT_ORDER_PLOT]

    ct_counts = ct.copy()
    total_n_cells = ct_counts.values.sum()   # for global % on segments

    # --- STEP 2b: overall agreement / disagreement for THIS QUESTION ---
    s = df[likert_col].dropna()              # exclude NA
    total_valid = s.shape[0]
    agree_n = s.isin(AGREE).sum()
    disagree_n = s.isin(DISAGREE).sum()

    agree_pct = agree_n / total_valid * 100 if total_valid > 0 else 0
    disagree_pct = disagree_n / total_valid * 100 if total_valid > 0 else 0

    summary_text = (
        f"N = {total_valid}\n"
        f"Agree: {agree_pct:.1f}% (n={agree_n})\n"
        f"Disagree: {disagree_pct:.1f}% (n={disagree_n})"
    )

    # --- STEP 3: plot COUNTS (like your screenshot) ---
    ax = ct_counts.plot(
        kind="barh",
        stacked=True,
        figsize=figsize
    )

    ax.set_xlabel("Percentage")
    ax.set_ylabel(checkbox_col)
    ax.set_title(f"{title_prefix}{likert_col}")

    # global-% labels on each segment (optional – this is what you had)
    for i, row in enumerate(ct_counts.values):
        cum = 0
        for count in row:
            if count > 0:
                pct_global = count / total_n_cells * 100
                ax.text(
                    cum + count / 2,
                    i,
                    f"{pct_global:.1f}%",
                    ha="center", va="center",
                    fontsize=8,
                    color="white" if count > 3 else "black"
                )
            cum += count

    # legend outside
    ax.legend(
        title="Response",
        bbox_to_anchor=(1.35, 1),
        loc="upper left"
    )

    # --- STEP 4: add summary box BESIDE the plot ---
    ax.text(
        1.02, 0.3,                   # x, y in axes coords (just outside right)
        summary_text,
        transform=ax.transAxes,
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray")
    )

    plt.tight_layout()
    plt.show()


In [ ]:

demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}



arrival_cols = [
    "Upon arrival, I prefer to interact with a human staff member rather than a digital system",
    "It feels easier to know what to do when a person guides me upon arrival",
    "Personal interaction with staff at arrival makes the dining experience feel more genuine",
]

# arrival_cols = [
#     "Upon arrival, I prefer to interact with a human staff member rather than a digital system",
#     "It feels easier to know what to do when a person guides me upon arrival",
#     "Personal interaction with staff at arrival makes the dining experience feel more genuine"
# ]

# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in arrival_cols:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Arrival in a restaurant and greeting by human – {demo_label} • "
        )



# Age band
# 25–34    33
# 35–44    11
# 45–54     6
# 55–64     3
# 18–24     3
# 65+       2
# Name: count, dtype: int64

# Most people strongly agree that Upon arrival, I prefer to interact with a human staff member rather than a digital system and everyone from 18-24 agree they want this


In [ ]:


# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band",
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}



meal_assistance = [
    "Being checked on by human staff during the meal makes the experience feel more personal",
    "It is easier to ask follow-up questions about dishes when speaking to a person",
    "It is easier to get assistance during the meal from human staff than from a digital system",
    "Human staff are better at judging the right time to bring the next course than digital systems",
    "It is easier to change or adjust my order when interacting with human staff",
    "I feel reassured when a person confirms my customized order verbally",
    "I like having the option of human assistance while using a digital system",
    "When a digital system fails, having human staff as a fallback makes me feel supported"
]




# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in meal_assistance:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Meal_assistance {demo_label} • "
        )

In [ ]:
# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}

# Despite imperfect wording (which is comparitive), people interpreted these three questions in a very consistent way. Ofcourse the definition of consistency
# here is subjective to interpretation...this is a limitation

# 3) Human interaction during the MEAL – HUMAN
Percived_value_of_human_led_menu_explaination = [
    "I trust human staff more than a digital system to provide accurate menu information", 
    "I enjoy hearing about dishes from a person rather than reading them on a screen"
]


# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in Percived_value_of_human_led_menu_explaination:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Percived_value_of_human_led_menu_explaination – {demo_label} • "
        )

In [ ]:
# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}

# Despite imperfect wording (which is comparitive), people interpreted these three questions in a very consistent way. Ofcourse the definition of consistency
# here is subjective to interpretation...this is a limitation

# 3) Human interaction during the MEAL – HUMAN
human_fallback_for_support = [
    "It is easier to clarify billing or payment questions with a person",
    "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system",
    "It is easier to ask follow-up questions about dishes when speaking to a person"
]

# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in human_fallback_for_support:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"human fallback for support – {demo_label} • "
        )

In [ ]:
# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}

# Despite imperfect wording (which is comparitive), people interpreted these three questions in a very consistent way. Ofcourse the definition of consistency
# here is subjective to interpretation...this is a limitation

# 3) Human interaction during the MEAL – HUMAN
Trust_in_humans_for_payment  = [
    "I trust human staff to handle payments more accurately than a digital system",
    "It is easier to understand the bill when a person explains what I am paying for",
    "Having a person handle payment makes me feel more confident and reassured about the transaction",
    "I feel more in control of my payment when interacting with a person rather than a digital system"
]


# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in Trust_in_humans_for_payment:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Trust_in_humans_for_payment – {demo_label} • "
        )

In [ ]:
# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}

# Despite imperfect wording (which is comparitive), people interpreted these three questions in a very consistent way. Ofcourse the definition of consistency
# here is subjective to interpretation...this is a limitation

# 3) Human interaction during the MEAL – HUMAN

Menu_exploration_with_humans = [
    "It is easier to understand menu items when a person explains them",
    #"I like when a staff member helps me explore the menu or suggests dishes",
    "I trust human staff more than a digital system to provide accurate menu information",
    "I enjoy hearing about dishes from a person rather than reading them on a screen"
]


# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in Menu_exploration_with_humans:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Menu_exploration_with_humans – {demo_label} • "
        )

In [ ]:
# Your demographic checkbox / category columns
demographic_checkboxes = {
    "Age band": "Age band"
    # "Dining frequency ( Select all that applies)": "Dining frequency",
    # "Typical group size while dining (Select all that applies)": "Typical group size",
    # "Which type of restaurant setting do you most often visit?\xa0 (Select all that applies)": "Restaurant setting",
}

# Despite imperfect wording (which is comparitive), people interpreted these three questions in a very consistent way. Ofcourse the definition of consistency
# here is subjective to interpretation...this is a limitation

# 3) Human interaction during the MEAL – HUMAN

Human_payment = [
    "I trust human staff to handle payments more accurately than a digital system",
    "It is easier to understand the bill when a person explains what I am paying for",
    "Having a person handle payment makes me feel more confident and reassured about the transaction",
    "I feel more in control of my payment when interacting with a person rather than a digital system"
]

# Dynamic loop for all demographics × all Likert items
for checkbox_col, demo_label in demographic_checkboxes.items():
    for q in Human_payment:
        plot_checkbox_vs_likert(
            df,
            checkbox_col=checkbox_col,
            likert_col=q,
            title_prefix=f"Human_payment – {demo_label} • "
        )

In [ ]:
# # # Create a dictionary mapping long question text → short labels
# # label_map = {
# #     "Upon arrival, I prefer to interact with a human staff member rather than a digital system": "Arrival_Pref",
# #     "It feels easier to know what to do when a person guides me upon arrival": "Guided_Ease",
# #     "Personal interaction with staff at arrival makes the dining experience feel more genuine": "Arrival_Genuine",
# #     "It is easier to understand menu items when a person explains them": "Menu_Understand",
# #     "I like when a staff member helps me explore the menu or suggests dishes": "Menu_Help",
# #     "I trust human staff more than a digital system to provide accurate menu information": "Menu_Trust",
# #     "I enjoy hearing about dishes from a person rather than reading them on a screen": "Menu_Hear",
# #     "When a digital system fails, having human staff as a fallback makes me feel supported": "Fallback_Support",
# #     "I like having the option of human assistance while using a digital system": "Option_Help",
# #     "It is easier to explain my dietary preferences or special needs to a person": "Dietary_Explain",
# #     "It is easier to change or adjust my order when interacting with human staff": "Order_Adjust",
# #     "I feel reassured when a person confirms my customized order verbally": "Order_Reassured",
# #     "It is easier to get assistance during the meal from human staff than from a digital system": "Meal_Assist",
# #     "Human staff are better at judging the right time to bring the next course than digital systems": "Timing_Judgement",
# #     "I feel more comfortable communicating personal or dietary needs to a person than through a digital device": "Comfort_Dietary",
# #     "Being checked on by human staff during the meal makes the experience feel more personal": "Checked_On",
# #     "It is easier to ask follow-up questions about dishes when speaking to a person": "FollowUp_Questions",
# #     "I trust human staff to handle payments more accurately than a digital system": "Pay_Trust",
# #     "It is easier to clarify billing or payment questions with a person": "Pay_Clarify",
# #     "If a payment issue occurs, I believe human staff can resolve it more easily than a digital system": "Pay_Issue",
# #     "It is easier to understand the bill when a person explains what I am paying for": "Bill_Understand",
# #     "Having a person handle payment makes me feel more confident and reassured about the transaction": "Pay_Confident",
# #     "I feel more in control of my payment when interacting with a person rather than a digital system": "Pay_Control"
# # }

# # corr_short = corr.rename(index=label_map, columns=label_map)
# # corr_short


# # plt.figure(figsize=(14, 12))

# # plt.imshow(corr_short, cmap='coolwarm', vmin=-1, vmax=1)
# # plt.colorbar(label="Spearman Correlation")

# # # Set axis labels
# # plt.xticks(np.arange(len(corr_short.columns)), corr_short.columns, rotation=90, fontsize=8)
# # plt.yticks(np.arange(len(corr_short.index)), corr_short.index, fontsize=8)

# # plt.title("Heatmap of Spearman Correlations (Short Labels)", fontsize=14)
# # plt.tight_layout()
# # plt.show()



# from scipy.stats import binomtest, kruskal


# # clean text (you already mostly do this)
# df_clean = df.copy()
# for col in df_clean.columns:
#     if df_clean[col].dtype == "object":
#         df_clean[col] = df_clean[col].str.strip().str.lower()




# def overall_agree_disagree(row, cols):
#     vals = row[cols].dropna()
#     if vals.empty:
#         return np.nan
    
#     agree_count = vals.isin(AGREE).sum()
#     disagree_count = vals.isin(DISAGREE).sum()
    
#     if (agree_count == 0) and (disagree_count == 0):
#         return np.nan  # all neutral
#     if agree_count > disagree_count:
#         return "agree"
#     elif disagree_count > agree_count:
#         return "disagree"
#     else:
#         return np.nan  # tie -> drop from test


# df_clean["arrival_overall"] = df_clean.apply(
#     overall_agree_disagree, cols=arrival_cols, axis=1
# )

# # counts
# arrival_counts = df_clean["arrival_overall"].value_counts(dropna=True)
# n_agree = int(arrival_counts.get("agree", 0))
# n_disagree = int(arrival_counts.get("disagree", 0))
# n_total = n_agree + n_disagree

# print("Agree:", n_agree, "Disagree:", n_disagree, "Total used:", n_total)

# # one-sided binomial test: H0: p_agree <= 0.5 vs H1: p_agree > 0.5
# if n_total > 0:
#     res = binomtest(n_agree, n_total, p=0.5, alternative="greater")
#     print(res)
# else:
#     print("Not enough data (no agree/disagree cases).")


In [ ]:
# 2. Make sure they are numeric Likert codes (e.g. 1–6)
# If you already coded them earlier, you can skip this part.
LIKERT_MAP = {
    "strongly disagree": 1,
    "disagree": 2,
    "slightly disagree": 3,
    "slightly agree": 4,
    "agree": 5,
    "strongly agree": 6,
}

for col in arrival_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.lower()
        .map(LIKERT_MAP)
    )

# 3. Sum the three items into a construct score (still ordinal, just more levels)
df["arrival_score"] = df[arrival_cols].sum(axis=1)

In [ ]:
df["arrival_score"]

In [ ]:
# Define the age order EXACTLY as it appears in your data
age_order = pd.CategoricalDtype(
    categories=[
        "18-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+",
    ],
    ordered=True,
)

df["age_band_ord"] = df["Age band"].astype(age_order)

# Convert ordered categories to integer codes 1..6
# 18-24 -> 0, 25-34 -> 1, ... so add 1 to start from 1
df["age_code"] = df["age_band_ord"].cat.codes + 1


In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel
import statsmodels.api as sm

# 1. Dependent variable as ordered categories (e.g. 1–6)
y = data["likert_code"]   # already numeric ordered codes

# 2. Explanatory variables WITHOUT constant
X = data[["age_code"]]    # or data["age_code"] if it's a Series

# 3. Fit the ordinal logistic regression
model = OrderedModel(
    y,
    X,
    distr="logit"   # or "probit"
)

res = model.fit(method="bfgs", disp=False)

print(res.summary())


In [ ]:
from scipy.stats import kruskal


# Type 3
# 	1. Based on plot 4 and 5 there exists a small percentage of population that do not feel reassured when humans reconfirm customize orders
# 	H0 = people across all age bands feel equally reassured by order conformation from humans
# 	H1 = People across age bands are not equally reassured by order confirmation from humans
	
# 	H0 = People across all age bands find it easy to customize the order when speaking to a human
# 	H1= People across different age bands do not feel equally reassured for adjusting their order when speaking with a human staff


# meal_assistance = [
#     "Being checked on by human staff during the meal makes the experience feel more personal",
#     "It is easier to get assistance during the meal from human staff than from a digital system",
#     "Human staff are better at judging the right time to bring the next course than digital systems",
#     "It is easier to change or adjust my order when interacting with human staff",
#     "I feel reassured when a person confirms my customized order verbally"
# ]



def kruskal_by_age(df, item_col,
                   age_col="Age band",
                   age_order=("18-24", "25-34", "35-44", "45-54", "55-64", "65+"),
                   min_n_per_group=2):
    """
    Kruskal–Wallis test of a single Likert item across ordered age bands.
    df        : pandas DataFrame
    item_col  : column name of the Likert item (string)
    age_col   : column with age bands
    age_order : order of age bands
    min_n_per_group : minimum N per age band to include in test
    """
    groups = []
    labels = []

    for age in age_order:
        mask = df[age_col] == age
        if not mask.any():
            continue

        # Get raw values for that age
        vals_raw = df.loc[mask, item_col].dropna()

        if vals_raw.empty:
            continue

        # Map Likert text → numeric
        vals_num = (
            vals_raw.astype(str)
                    .str.strip()
                    .str.lower()
                    .map(LIKERT_MAP)
        )

        vals_num = vals_num.dropna()

        if len(vals_num) >= min_n_per_group:
            groups.append(vals_num.values)
            labels.append(age)

    print(f"\nItem: {item_col}")
    print("Age groups included and Ns:")
    for lab, g in zip(labels, groups):
        print(f"  {lab}: N = {len(g)}")

    if len(groups) < 2:
        print("Not enough age groups with data to run Kruskal–Wallis.")
        return None

    stat, p = kruskal(*groups)
    print(f"\nKruskal–Wallis H = {stat:.3f}, p = {p:.4f}")
    if p < 0.05:
        print("→ Reject H₀: distributions differ across age bands.")
    else:
        print("→ Fail to reject H₀: no clear age effect detected.")

    return stat, p



In [ ]:

Item4 = "It is easier to change or adjust my order when interacting with human staff"                                                   
item5 = "I feel reassured when a person confirms my customized order verbally"

kruskal_by_age(df, Item4)
kruskal_by_age(df, item5)


In [ ]:
df.groupby("Age band")[Item4].apply(lambda s: s.notna().sum())
df.groupby("Age band")[item5].apply(lambda s: s.notna().sum())


In [ ]:
from scipy.stats import mannwhitneyu

age_col = "Age_3grp"

pairs = [
    ("18–34", "35–54"),
    ("18–34", "55+"),
    ("35–54", "55+")
]

for q in likert_cols:

    print(f"\n--- {q} ---")

    for g1, g2 in pairs:

        x = df_human_answered.loc[df_human_answered[age_col] == g1, q].dropna()
        y = df_human_answered.loc[df_human_answered[age_col] == g2, q].dropna()

        if len(x) == 0 or len(y) == 0:
            print(f"{g1} vs {g2}: NOT ENOUGH DATA")
            continue

        U, p = mannwhitneyu(x, y, alternative="two-sided")

        print(f"{g1} vs {g2} → p = {p:.4f}")
